# 최종 Test 평가

## 평가 원칙

- Baseline → V1 → V2의 개선 방향과 최종 후보 선택은 **Validation에서 모두 완료**
- V2를 최종 후보로 고정한 뒤 **Test Set은 마지막 일반화 성능 확인에만 사용**
- Test 결과를 보고 추가 학습, 하이퍼파라미터 변경, 모델 재선택을 수행하지 않음
- Baseline/V1/V2 Test 결과는 개선 과정의 사후 확인용 비교이며, V2 선택 근거는 Validation 결과
- Test Set: **733 images / 749 objects**

> 아래 실행은 세 모델에 동일한 `conf=0.25` 조건을 적용한 상대 비교입니다.  
> Ultralytics의 표준 PR 기반 mAP benchmark는 낮은 confidence 기본값을 사용하는 방식이 일반적이므로, 본 표의 수치는 **동일 조건에서의 모델 간 비교**로 해석합니다.

In [3]:
from pathlib import Path

import pandas as pd
from IPython.display import display


ROOT = Path.cwd().parent if Path.cwd().name == "analysis" else Path.cwd()
MODEL_NAMES = ["baseline", "improved_v1", "improved_v2"]


# 모델별 가장 최근 Test 결과 불러오기
rows = []

for model_name in MODEL_NAMES:
    model_dir = ROOT / "models" / model_name

    metric_files = sorted(
        model_dir.glob("test_*/test_metrics.csv"),
        key=lambda path: path.parent.name,
    )

    if not metric_files:
        raise FileNotFoundError(
            f"{model_name}의 Test 결과가 없습니다.\n"
            f"먼저 src/06_test.py를 실행하세요."
        )

    metric_path = metric_files[-1]
    metrics = pd.read_csv(metric_path).iloc[0].to_dict()

    metrics["Model"] = model_name
    metrics["Test Result"] = metric_path.parent.name
    rows.append(metrics)

    print(
        f"{model_name:12} | "
        f"{metric_path.relative_to(ROOT).as_posix()}"
    )


# 모델 비교표 생성
columns = [
    "Model",
    "Precision",
    "Recall",
    "F1",
    "mAP50",
    "mAP75",
    "mAP50-95",
    "Inference(ms)",
]

test_df = pd.DataFrame(rows)[columns]

print("\n최종 Test 결과")
display(test_df.round(4))

baseline     | models/baseline/test_260904_0004/test_metrics.csv
improved_v1  | models/improved_v1/test_260904_0005/test_metrics.csv
improved_v2  | models/improved_v2/test_260904_0008/test_metrics.csv

최종 Test 결과


,Model,Precision,Recall,F1,mAP50,mAP75,mAP50-95,Inference(ms)
0,baseline,0.9483,0.8838,0.9149,0.9340,0.7187,0.6269,2.1254
1,improved_v1,0.9661,0.9252,0.9452,0.9687,0.7624,0.6685,3.3289
2,improved_v2,0.9704,0.9372,0.9535,0.9584,0.7954,0.6956,3.3466


## 1. Test 성능 비교

| Model | Image Size | Precision | Recall | F1 | mAP50 | mAP75 | mAP50-95 | Inference(ms) |
|---|---:|---:|---:|---:|---:|---:|---:|---:|
| Baseline | 640 | 0.9483 | 0.8838 | 0.9149 | 0.9340 | 0.7187 | 0.6269 | 2.13 |
| Improved V1 | 960 | 0.9661 | 0.9252 | 0.9452 | **0.9687** | 0.7624 | 0.6685 | 3.33 |
| Improved V2 | 960 | **0.9704** | **0.9372** | **0.9535** | 0.9584 | **0.7954** | **0.6956** | 3.35 |

## 2. 최종 판단

### Validation 기반 개선 결과

| 단계 | 변경 | 주요 결과 |
|---|---|---|
| Baseline | Input Size 640 / Epoch 30 | Q1 Recall 0.7489 / FN 123 / FP 99 |
| V1 | Input Size 640 → 960 | Q1 Recall 0.8265 / FN 80 / FP 75 |
| V2 | Epoch 30 → 100 | Recall 0.9032 / mAP75 0.7703 / FN 76 / FP 51 |

### 개선 과정

- **V1**: Input Size 증가를 통한 극소형 UAV 검출 개선
  - Q1 Recall **0.7489 → 0.8265**
  - FN **123 → 80**
  - FP **99 → 75**

- **V2**: Input Size 유지 후 Epoch 증가를 통한 추가 학습
  - Recall **0.8847 → 0.9032**
  - mAP75 **0.7225 → 0.7703**
  - mAP50-95 **0.6202 → 0.6500**
  - FN **80 → 76**
  - FP **75 → 51**
  - mAP50은 **0.9362 → 0.9354**로 유사 수준 유지

Validation 결과를 기준으로 **V2를 최종 모델로 고정**

### 최종 Test 확인

| Metric | Baseline | V2 | 변화 |
|---|---:|---:|---:|
| Precision | 0.9483 | **0.9704** | +0.0221 |
| Recall | 0.8838 | **0.9372** | +0.0534 |
| F1 | 0.9149 | **0.9535** | +0.0386 |
| mAP50 | 0.9340 | **0.9584** | +0.0244 |
| mAP75 | 0.7187 | **0.7954** | +0.0767 |
| mAP50-95 | 0.6269 | **0.6956** | +0.0687 |

- Validation에서 확인한 **Recall 및 bbox 정밀도 개선이 Test에서도 유지**
- Test에서는 객체 크기별·실패 유형별 추가 분석 미수행
- Test 결과를 이용한 모델 재선택 및 추가 튜닝 미수행

**Test Set은 Validation에서 선택한 V2의 최종 일반화 성능 확인에만 사용**

---

## 3. 잔여 오류 및 향후 개선 방향

잔여 오류는 **V2 Validation 실패 분석 결과**를 기준으로 정리

| 잔여 문제 | 확인 결과 | 향후 개선 방향 |
|---|---|---|
| 극소형 UAV | Q1 Recall 0.8311로 가장 낮음 | 극소형·저화질 UAV 학습 데이터 추가 |
| 미검출 | FN 76개 중 41개 | 복잡한 배경·낮은 대비의 어려운 UAV 사례 추가 |
| 낮은 신뢰도 | FN 25개 | Validation 기준 confidence threshold 재검토 |
| 배경 오탐 | 조류·비행기·건물·바위·그림자 등과 혼동 | 모델이 UAV로 잘못 인식한 배경 사례 추가 |
| bbox 위치 오차 | 위치 부정확 FN 10개 | GT annotation 일관성 점검 및 bbox 정밀도 개선 |
| 일반화 검증 범위 | 동일 데이터셋 내 Validation·Test에서 성능 확인 | 다른 촬영 환경 데이터에 대한 추가 검증 필요 |

### 개선 우선순위

1. **UAV로 잘못 인식한 배경 사례 추가**
2. **극소형·저화질 UAV 및 복잡한 배경 데이터 보강**
3. **Validation 기반 confidence threshold 재검토**
4. **다른 촬영 환경의 데이터로 일반화 성능 검증**

추가 Epoch 증가보다 **실제 잔여 실패 사례를 학습 데이터에 반영하는 개선 우선**

### 최종 결론

**Baseline 실패 분석 → Input Size 증가(V1) → 극소형 UAV 개선 → Epoch 증가(V2) → Validation에서 최종 모델 고정 → Test 최종 확인**

- Baseline 대비 **모든 주요 Test 지표 개선**
- 극소형 UAV Recall 개선 및 전체 FN·FP 감소
- 높은 IoU 조건의 bbox 성능 개선
- 극소형·저화질 UAV 미검출과 배경 유사 객체 오탐은 잔여 한계
- 향후 개선 방향은 **추가 학습보다 실패 사례 중심의 데이터 보강**